# 06 — Mapping the 25 Atlanta study ZCTAs and understanding CBGs

**What are we doing?** We draw the *actual Census polygons* for all 25 study ZIP-like areas, then zoom in to show how smaller Census Block Groups overlap two selected ZCTAs.

**Important:** Postal ZIP codes and Census ZCTAs are not identical. The map uses Census **ZCTA** boundaries, because ACS statistics are reported at the ZCTA level. This is a geographic illustration, not proof that the vendor’s source-CBG vintage has been validated.

In [ ]:
from pathlib import Path
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
GEO = ROOT / "data" / "public_geography"
FIG = ROOT / "results" / "figures"
FIG.mkdir(parents=True, exist_ok=True)

zcta = gpd.read_file(GEO / "atlanta_25_zcta_boundaries.geojson")
cbg = gpd.read_file(GEO / "host_zcta_intersecting_cbg_boundaries.geojson")
assert len(zcta) == 25
print("Study ZCTAs:", ", ".join(sorted(zcta.ZCTA)))
print("Local CBG polygons:", len(cbg))

## Figure 1 — All 25 measured ZCTAs

Each blue polygon is a Census ZCTA included in the study. Orange polygons are the **host ZCTAs** used in the Lenox vs Greenbriar neighborhood-income comparison. The map does **not** show every ZIP code in Atlanta.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))
zcta.plot(ax=ax, facecolor="#dbeafe", edgecolor="#334155", linewidth=1)
zcta[zcta.ZCTA.isin(["30326", "30331"])].plot(ax=ax, facecolor="#f59e0b", edgecolor="#92400e", linewidth=2)
for _, row in zcta.iterrows():
    p = row.geometry.representative_point()
    ax.text(p.x, p.y, row.ZCTA, fontsize=8, ha="center", va="center", bbox=dict(facecolor="white", alpha=.7, edgecolor="none", pad=1))
ax.set_title("Atlanta: 25 study ZCTAs")
ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
ax.set_aspect("equal")
fig.savefig(FIG / "atlanta_25_study_zctas_reproduced.png", dpi=180, bbox_inches="tight")
plt.show()

## Figure 2 — ZCTA versus Census Block Group

A **ZCTA** approximates a postal ZIP area for Census statistics. A **CBG** is a smaller statistical area nested within a Census tract. Their boundaries do not always line up. A single CBG can intersect more than one ZCTA. We use this overlay to explain why a crosswalk is needed, **not** to claim the 2024 CBG polygons match the proprietary mobility source geography.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 7))
for ax, code, name in zip(axes, ["30326", "30331"], ["Lenox", "Greenbriar"]):
    area = zcta[zcta.ZCTA == code]
    local = cbg[cbg.intersects(area.geometry.iloc[0])]
    local.plot(ax=ax, facecolor="#a5f3fc", edgecolor="#0891b2", linewidth=.6)
    area.plot(ax=ax, facecolor="none", edgecolor="#ea580c", linewidth=3)
    xmin, ymin, xmax, ymax = area.total_bounds
    ax.set_xlim(xmin-(xmax-xmin)*.1, xmax+(xmax-xmin)*.1)
    ax.set_ylim(ymin-(ymax-ymin)*.1, ymax+(ymax-ymin)*.1)
    ax.set_title(f"{name}: ZCTA {code}")
    ax.set_aspect("equal")
fig.legend(handles=[Patch(facecolor="#a5f3fc", edgecolor="#0891b2", label="Census Block Groups"), Patch(facecolor="white", edgecolor="#ea580c", linewidth=2, label="ZCTA boundary")], loc="lower center", ncol=2)
fig.suptitle("ZCTA boundaries versus CBG boundaries")
fig.tight_layout(rect=(0,.08,1,.94))
fig.savefig(FIG / "zcta_vs_cbg_reproduced.png", dpi=180, bbox_inches="tight")
plt.show()

## How the crosswalk fits the regression

For origin CBG $b$, destination mall-area $j$, month $t$, and destination ZCTA $z$:

$$Y_{zjt}=\sum_b w_{bz}Y_{bjt}$$

$w_{bz}$ is the share assigned from CBG $b$ to ZCTA $z$. We then merge ZCTA-level income, Gini, population, and distance into the regression panel. **This map does not estimate $w_{bz}$**. Our earlier crosswalks remain provisional and must be checked against the mobility vendor’s CBG vintage and suppression rules.

## Data provenance and limits

- **ZCTA geometry:** U.S. Census Bureau, TIGER/Line 2026 ZCTA5 file, using 2020 ZCTA definitions (`ZCTA5CE20`).
- **CBG geometry:** U.S. Census Bureau, TIGER/Line 2024 Georgia Block Groups.
- Only 25 selected ZCTAs and local CBG geometries are included in this public repository.
- These boundary products are for visualization, not a validated vendor-data crosswalk.
- The host-ZCTA map does not represent an equal-radius neighborhood or mall parcel boundary.